# 13. Linear regression and numerical prediction

**Level:** Intermediate

**Prerequisites:** 10-12: losses, splits, pipelines.

[Course index](00_START_HERE.ipynb) | [Previous](12_Preprocessing_and_Leakage_Safe_Pipelines.ipynb) | [Next](14_Logistic_Regression.ipynb)

**Topics:** Linear regression, Regression, Coefficients, MAE, Ridge regression.

**Matching script:** `examples\13_Linear_Regression.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Linear regression predicts a numeric target as a weighted sum of features plus an intercept.
Least squares chooses coefficients to minimize squared training error. 'Linear' refers to
coefficients: polynomial input features can still be used in a linear regression model.

Interpret a coefficient in its feature's units while holding other modeled features fixed.
Correlated features can make coefficients unstable, and a predictive association is not a causal
effect. Extrapolating outside the training feature range can be unreliable.

Ridge adds an L2 penalty that shrinks coefficients. It can reduce variance at the cost of bias,
especially when features are correlated or numerous. Scale features when the regularization
penalty should treat their units comparably. Compare against a simple mean prediction baseline
on an untouched test set rather than celebrating a low training error.

## Mathematics or data contract

y_hat = Xw+b; least-squares minimizes sum((y-y_hat)^2). Ridge adds alpha*sum(w^2).

## Important limitation

Synthetic linear data deliberately matches the model family; real latency may be nonlinear and time-dependent.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Held-out regression versus a constant baseline

**Purpose:** Fit an actual estimator and evaluate it on unseen synthetic rows.

### Read the code in this order
1. Generate a noisy linear target with a fixed seed.
2. Fit only the training partition.
3. Compute MAE on untouched test rows.
4. Require improvement over a training-mean baseline.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
coefficients: [0.303, 0.099]
test MAE: 0.892 baseline MAE: 7.544
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
X = rng.uniform(0, 100, (240, 2))
y = 5 + X @ np.array([0.3, 0.1]) + rng.normal(0, 1, 240)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LinearRegression().fit(X_train, y_train)
error = mean_absolute_error(y_test, model.predict(X_test))
baseline = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))
assert error < baseline and error < 2
print("coefficients:", np.round(model.coef_, 3).tolist())
print("test MAE:", round(error, 3), "baseline MAE:", round(baseline, 3))

coefficients: [0.303, 0.099]
test MAE: 0.892 baseline MAE: 7.544


### Interpret example 1

The result should demonstrate this purpose: **Fit an actual estimator and evaluate it on unseen synthetic rows.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Learn a slope with gradient descent

**Purpose:** Connect the library estimator to a transparent optimization algorithm.

### Read the code in this order
1. Generate a known linear relationship.
2. Calculate the analytical batch gradient.
3. Apply bounded updates and verify the learned slope against the known value.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
learned slope: 3.0 loss decreased: True
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import numpy as np

X = np.array([0., 0.25, 0.5, 0.75, 1.])
y = 3 * X
weight = 0.0
initial = np.mean((weight * X - y) ** 2)
for _ in range(160):
    gradient = 2 * np.mean(X * (weight * X - y))
    weight -= 0.2 * gradient
final = np.mean((weight * X - y) ** 2)
assert final < initial and abs(weight - 3) < 1e-6
print("learned slope:", round(weight, 6), "loss decreased:", final < initial)

learned slope: 3.0 loss decreased: True


### Interpret example 2

The result should demonstrate this purpose: **Connect the library estimator to a transparent optimization algorithm.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add nonlinear congestion and compare linear regression with a tree using the same held-out split.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Linear regression` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: y_hat = Xw+b; least-squares minimizes sum((y-y_hat)^2). Ridge adds alpha*sum(w^2).
- Name the boundary of the demonstration: Synthetic linear data deliberately matches the model family; real latency may be nonlinear and time-dependent.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.